In [ ]:
# Step 1: match_sop(Failure reported during May 27, 2020 from 01:00 to 01:30 (UTC+8). Need to identify root cause component, exact occurrenc
...[truncated])

```
Matched SOPs:
- RCA-Agent Metric Fault Identification SOP (score 0.49)
  Name: RCA-Agent Metric Fault Identification SOP
Steps:
1. whether_is_abnormal_metric(start_time, end_time, '', None): scan all candidate components and KPIs for compact anomaly evidence and onset times.
2. component_analyze(start_time, end_time): compare the metric evidence with trace/log anomaly counts for the same window.
3. The answer is the observations obtained from the former steps.
- RCA-Agent Log-Based Reason Analysis SOP (score 0.43)
  Name: RCA-Agent Log-Based Reason Analysis SOP
Steps:
1. get_logs(None, start_time, end_time): summarize log anomalies and raw error messages for candidate components in the incident window.
2. get_relevant_metric('error'): list error, resource, and traffic-related metric fields that may explain the logs.
3. whether_is_abnormal_metric(start_time, end_time, '', None): compare log evidence against metric anomalies and onset times.
4. The answer is the observations obtained from the former steps.
- RCA-Agent Trace Localization SOP (score 0.41)
  Name: RCA-Agent Trace Localization SOP
Steps:
1. collect_trace(start_time, end_time): summarize abnormal call duration, error-rate, traffic-drop, and downstream information in the incident window.
2. whether_is_abnormal_metric(start_time, end_time, '', None): confirm which trace-localized components are also faulty in metrics.
3. The answer is the observations obtained from the former steps.
```

In [ ]:
# Step 2: generate_sop_code for 'RCA-Agent Metric Fault Identification SOP'
start_time = '2020-05-27 01:00:00'
end_time = '2020-05-27 01:30:00'
metrics_obs = whether_is_abnormal_metric(start_time, end_time, '', None)
analysis = component_analyze(start_time, end_time)
answer = metrics_obs + "\n" + analysis

```
Generated code for SOP 'RCA-Agent Metric Fault Identification SOP':
start_time = '2020-05-27 01:00:00'
end_time = '2020-05-27 01:30:00'
metrics_obs = whether_is_abnormal_metric(start_time, end_time, '', None)
analysis = component_analyze(start_time, end_time)
answer = metrics_obs + "\n" + analysis
```

In [ ]:
# Step 3: run_sop
start_time = '2020-05-27 01:00:00'
end_time = '2020-05-27 01:30:00'
metrics_obs = whether_is_abnormal_metric(start_time, end_time, '', None)
analysis = component_analyze(start_time, end_time)
answer = metrics_obs + "\n" + analysis

```
[ANOMALY] db_007 DbTime: rise to 26.7 at 2020-05-27 01:12:00 (median 6.2, thr 16.3, sev 23.0, n=1)
[ANOMALY] db_007 Exec_Per_Sec: rise to 95.9 at 2020-05-27 01:12:00 (median 39.05, thr 79.26, sev 17.0, n=2)
[ANOMALY] db_003 Exec_Per_Sec: rise to 242.2 at 2020-05-27 01:12:00 (median 106.2, thr 164.6, sev 7.0, n=1)
[ANOMALY] db_007 DFParaWrite_Per_Sec: rise to 2.21 at 2020-05-27 01:08:00 (median 1.285, thr 1.93, sev 4.3, n=2)
Component anomaly overview (ranked):
docker_002  metric_anoms=0 (top: n/a sev=0.0)  trace_anoms=21  log_anoms=0
os_021  metric_anoms=12 (top: Disk_await sev=78.0)  trace_anoms=7  log_anoms=0
os_018  metric_anoms=16 (top: Disk_io_util sev=70.4)  trace_anoms=0  log_anoms=0
db_007  metric_anoms=16 (top: Redo_Per_Sec sev=66.8)  trace_anoms=0  log_anoms=0
docker_003  metric_anoms=1 (top: container_cpu_used sev=47.5)  trace_anoms=15  log_anoms=0
docker_006  metric_anoms=1 (top: container_cpu_used sev=46.2)  trace_anoms=11  log_anoms=0
docker_001  metric_anoms=0 (top: n/a sev=0.0)  trace_anoms=12  log_anoms=0
docker_004  metric_anoms=0 (top: n/a sev=0.0)  trace_anoms=12  log_anoms=0
db_003  metric_anoms=6 (top: Redo_Per_Sec sev=73.1)  trace_anoms=0  log_anoms=0
docker_008  metric_anoms=1 (top: container_cpu_used sev=7.4)  trace_anoms=5  log_anoms=0
docker_005  metric_anoms=0 (top: n/a sev=0.0)  trace_anoms=6  log_anoms=0
os_020  metric_anoms=5 (top: Disk_svctm sev=77.2)  trace_anoms=0  log_anoms=0
os_017  metric_anoms=5 (top: Disk_svctm sev=68.7)  trace_anoms=0  log_anoms=0
os_001  metric_anoms=5 (top: Disk_avgqu_sz sev=45.4)  trace_anoms=0  log_anoms=0
os_009  metric_anoms=2 (top: Send_total sev=5.4)  trace_anoms=0  log_anoms=0
```

In [ ]:
# Step 4: match_observation([ANOMALY] db_007 DbTime: rise to 26.7 at 2020-05-27 01:12:00 (median 6.2, thr 16.3, sev 23.0, n=1)
[ANOMALY] db_007 Exec
...[truncated])

```
ObAgent: uncertain.

Reason: DB time and Exec_Per_Sec rose while many OS nodes show strong disk-I/O anomalies (Disk_await, Disk_io_util, Disk_svctm) and DB redo/write rates increased — this points to storage I/O contention rather than a clear network loss/delay or connection-close/limit. To disambiguate, run:
- whether_is_abnormal_metric(os_<db-host>, Disk_io_util) and Disk_await/Disk_svctm
- whether_is_abnormal_metric(db_<host>, CPU_util or container_cpu_used)
- whether_is_abnormal_metric(net_if_<db-host>, tx_retrans/if_errors) to rule out network loss
- collect_trace of DB wait events (I/O vs CPU vs network) to confirm the root cause.
JudgeAgent: No - The trace/metric summary indicates storage I/O contention (disk I/O anomalies on multiple os_* nodes and DB redo/write/DbTime spikes at 2020-05-27 01:12:00 and 01:08:00), but that reason is not one of the allowed anomaly types (CPU fault, network delay, network loss, db connection limit, db close). Additionally the history does not pin a single component as the location (multiple os_ and db_ candidates). Thus the root cause is not correctly and completely identified.
```

In [ ]:
# Step 5: match_sop(Find SOPs related to storage I/O / disk contention, DB wait/trace localization, and network loss/delay for the incident 
...[truncated])

```
Matched SOPs:
- RCA-Agent Trace Localization SOP (score 0.45)
  Name: RCA-Agent Trace Localization SOP
Steps:
1. collect_trace(start_time, end_time): summarize abnormal call duration, error-rate, traffic-drop, and downstream information in the incident window.
2. whether_is_abnormal_metric(start_time, end_time, '', None): confirm which trace-localized components are also faulty in metrics.
3. The answer is the observations obtained from the former steps.
- RCA-Agent Metric Fault Identification SOP (score 0.44) (already executed)
  Name: RCA-Agent Metric Fault Identification SOP
Steps:
1. whether_is_abnormal_metric(start_time, end_time, '', None): scan all candidate components and KPIs for compact anomaly evidence and onset times.
2. component_analyze(start_time, end_time): compare the metric evidence with trace/log anomaly counts for the same window.
3. The answer is the observations obtained from the former steps.
- RCA-Agent Log-Based Reason Analysis SOP (score 0.37)
  Name: RCA-Agent Log-Based Reason Analysis SOP
Steps:
1. get_logs(None, start_time, end_time): summarize log anomalies and raw error messages for candidate components in the incident window.
2. get_relevant_metric('error'): list error, resource, and traffic-related metric fields that may explain the logs.
3. whether_is_abnormal_metric(start_time, end_time, '', None): compare log evidence against metric anomalies and onset times.
4. The answer is the observations obtained from the former steps.
```

In [ ]:
# Step 6: generate_sop_code for 'RCA-Agent Trace Localization SOP'
start_time = '2020-05-27 01:00:00'
end_time = '2020-05-27 01:30:00'
trace_obs = collect_trace(start_time, end_time, None)
metric_obs = whether_is_abnormal_metric(start_time, end_time, '', None)
answer = trace_obs + '\n' + metric_obs

```
Generated code for SOP 'RCA-Agent Trace Localization SOP':
start_time = '2020-05-27 01:00:00'
end_time = '2020-05-27 01:30:00'
trace_obs = collect_trace(start_time, end_time, None)
metric_obs = whether_is_abnormal_metric(start_time, end_time, '', None)
answer = trace_obs + '\n' + metric_obs
```

In [ ]:
# Step 7: run_sop
start_time = '2020-05-27 01:00:00'
end_time = '2020-05-27 01:30:00'
trace_obs = collect_trace(start_time, end_time, None)
metric_obs = whether_is_abnormal_metric(start_time, end_time, '', None)
answer = trace_obs + '\n' + metric_obs

```
[TRACE] docker_002 trace.self.jdbc.duration_p95: rise to 0.08205 at 2020-05-27 01:04:00 (median 0.003, sev 55.9)
[TRACE] docker_001 trace.self.flyremote.duration_p95: rise to 0.0478 at 2020-05-27 01:18:00 (median 0.0071, sev 45.2)
[TRACE] docker_002 trace.to_docker_006.remoteprocess.duration_p95: rise to 5.678 at 2020-05-27 01:27:00 (median 0.03538, sev 41.8)
[TRACE] docker_006 trace.from_docker_002.remoteprocess.duration_p95: rise to 5.678 at 2020-05-27 01:27:00 (median 0.03538, sev 41.8)
[TRACE] docker_006 trace.self.local.duration_mean: rise to 0.624 at 2020-05-27 01:27:00 (median 0.006001, sev 41.7)
[TRACE] docker_006 trace.self.local.duration_p95: rise to 1.497 at 2020-05-27 01:27:00 (median 0.009, sev 41.3)
[TRACE] os_022 trace.self.csf.duration_p95: rise to 14.4 at 2020-05-27 01:27:00 (median 1.383, sev 40.7)
[TRACE] os_022 trace.from_root.osb.duration_p95: rise to 14.42 at 2020-05-27 01:27:00 (median 1.403, sev 40.7)
[TRACE] docker_002 trace.self.csf.duration_p95: rise to 4.954 at 2020-05-27 01:27:00 (median 0.1067, sev 40.4)
[TRACE] docker_002 trace.self.csf.duration_mean: rise to 1.618 at 2020-05-27 01:27:00 (median 0.04991, sev 39.6)
[TRACE] docker_002 trace.from_os_022.remoteprocess.duration_mean: rise to 6.909 at 2020-05-27 01:27:00 (median 0.4941, sev 39.2)
[TRACE] os_022 trace.to_docker_002.remoteprocess.duration_mean: rise to 6.909 at 2020-05-27 01:27:00 (median 0.4941, sev 39.2)
[TRACE] docker_001 trace.to_docker_006.remoteprocess.duration_mean: rise to 2.642 at 2020-05-27 01:27:00 (median 0.02411, sev 38.7)
[TRACE] docker_006 trace.from_docker_001.remoteprocess.duration_mean: rise to 2.642 at 2020-05-27 01:27:00 (median 0.02411, sev 38.7)
[TRACE] docker_002 trace.to_docker_006.remoteprocess.duration_mean: rise to 2.647 at 2020-05-27 01:27:00 (median 0.02482, sev 38.6)
Most-downstream faulty component(s): docker_001, docker_002, docker_003, docker_004, docker_005, docker_006, docker_007, docker_008, os_021, os_022
[ANOMALY] db_007 DbTime: rise to 26.7 at 2020-05-27 01:12:00 (median 6.2, thr 16.3, sev 23.0, n=1)
[ANOMALY] db_007 Exec_Per_Sec: rise to 95.9 at 2020-05-27 01:12:00 (median 39.05, thr 79.26, sev 17.0, n=2)
[ANOMALY] db_003 Exec_Per_Sec: rise to 242.2 at 2020-05-27 01:12:00 (median 106.2, thr 164.6, sev 7.0, n=1)
[ANOMALY] db_007 DFParaWrite_Per_Sec: rise to 2.21 at 2020-05-27 01:08:00 (median 1.285, thr 1.93, sev 4.3, n=2)
```

In [ ]:
# Step 8: match_observation([ANOMALY] db_007 DbTime: rise to 26.7 at 2020-05-27 01:12:00 (median 6.2, thr 16.3, sev 23.0, n=1)
[ANOMALY] db_007 Exec
...[truncated])

```
ObAgent: uncertain

Analysis: traces show large end-to-end latency spikes across services while DB metrics (DbTime, Redo_Per_Sec, DFParaWrite) and OS disk KPIs (Disk_await, Disk_io_util, Disk_svctm) are high — this points to storage/DB performance issues rather than an obvious network loss or connection-close. To disambiguate, check:
- whether_is_abnormal_metric(os_018, Disk_io_util) and os_021 Disk_await (confirm IO bottleneck)
- whether_is_abnormal_metric(network, tcp_retransmits) and link RTT (rule out network loss/delay)
- whether_is_abnormal_metric(db_007, connections_active / connection_errors) and DB logs (rule out connection limit/close).
JudgeAgent: No - The agents point to storage I/O contention (e.g., db_007 DbTime/Exec_Per_Sec spikes at 2020-05-27 01:12:00 and widespread OS disk KPI anomalies, plus trace latency spikes ~01:27:00) but they did not pick exactly one component from the candidate list, the occurrence minute is ambiguous (01:12 vs 01:27), and the diagnosed cause (storage I/O / disk contention) is not one of the allowed anomaly types. Thus the root cause is not correctly and completely identified.
```

In [ ]:
# Step 9: match_sop(Find SOPs for storage I/O / DB wait localization and DB connection checks for the incident window 2020-05-27 01:00:00 to
...[truncated])

```
Matched SOPs:
- RCA-Agent Trace Localization SOP (score 0.43) (already executed)
  Name: RCA-Agent Trace Localization SOP
Steps:
1. collect_trace(start_time, end_time): summarize abnormal call duration, error-rate, traffic-drop, and downstream information in the incident window.
2. whether_is_abnormal_metric(start_time, end_time, '', None): confirm which trace-localized components are also faulty in metrics.
3. The answer is the observations obtained from the former steps.
- RCA-Agent Metric Fault Identification SOP (score 0.40) (already executed)
  Name: RCA-Agent Metric Fault Identification SOP
Steps:
1. whether_is_abnormal_metric(start_time, end_time, '', None): scan all candidate components and KPIs for compact anomaly evidence and onset times.
2. component_analyze(start_time, end_time): compare the metric evidence with trace/log anomaly counts for the same window.
3. The answer is the observations obtained from the former steps.
- RCA-Agent Preprocess and Threshold SOP (score 0.39)
  Name: RCA-Agent Preprocess and Threshold SOP
Steps:
1. component_analyze(start_time, end_time): summarize candidate components with metric, trace, and log anomaly counts inside the incident window.
2. get_relevant_metric(''): list available metric KPI names so later checks use valid fields rather than guessed names.
3. The answer is the observations obtained from the former steps.
```

In [ ]:
# Step 10: generate_sop_code for 'RCA-Agent Preprocess and Threshold SOP'
start_time = '2020-05-27 01:00:00'
end_time = '2020-05-27 01:30:00'
component_summary = component_analyze(start_time, end_time)
available_kpis = get_relevant_metric('')
answer = component_summary + '\n' + available_kpis

```
Generated code for SOP 'RCA-Agent Preprocess and Threshold SOP':
start_time = '2020-05-27 01:00:00'
end_time = '2020-05-27 01:30:00'
component_summary = component_analyze(start_time, end_time)
available_kpis = get_relevant_metric('')
answer = component_summary + '\n' + available_kpis
```

In [ ]:
# Step 11: run_sop
start_time = '2020-05-27 01:00:00'
end_time = '2020-05-27 01:30:00'
component_summary = component_analyze(start_time, end_time)
available_kpis = get_relevant_metric('')
answer = component_summary + '\n' + available_kpis

```
Component anomaly overview (ranked):
docker_002  metric_anoms=0 (top: n/a sev=0.0)  trace_anoms=21  log_anoms=0
os_021  metric_anoms=12 (top: Disk_await sev=78.0)  trace_anoms=7  log_anoms=0
os_018  metric_anoms=16 (top: Disk_io_util sev=70.4)  trace_anoms=0  log_anoms=0
db_007  metric_anoms=16 (top: Redo_Per_Sec sev=66.8)  trace_anoms=0  log_anoms=0
docker_003  metric_anoms=1 (top: container_cpu_used sev=47.5)  trace_anoms=15  log_anoms=0
docker_006  metric_anoms=1 (top: container_cpu_used sev=46.2)  trace_anoms=11  log_anoms=0
docker_001  metric_anoms=0 (top: n/a sev=0.0)  trace_anoms=12  log_anoms=0
docker_004  metric_anoms=0 (top: n/a sev=0.0)  trace_anoms=12  log_anoms=0
db_003  metric_anoms=6 (top: Redo_Per_Sec sev=73.1)  trace_anoms=0  log_anoms=0
docker_008  metric_anoms=1 (top: container_cpu_used sev=7.4)  trace_anoms=5  log_anoms=0
docker_005  metric_anoms=0 (top: n/a sev=0.0)  trace_anoms=6  log_anoms=0
os_020  metric_anoms=5 (top: Disk_svctm sev=77.2)  trace_anoms=0  log_anoms=0
os_017  metric_anoms=5 (top: Disk_svctm sev=68.7)  trace_anoms=0  log_anoms=0
os_001  metric_anoms=5 (top: Disk_avgqu_sz sev=45.4)  trace_anoms=0  log_anoms=0
os_009  metric_anoms=2 (top: Send_total sev=5.4)  trace_anoms=0  log_anoms=0
Available metric KPIs (first 30):
- ACS
- AIOS
- AWS
- CPU_Used_Pct
- CPU_free_pct
- Call_Per_Sec
- DFParaWrite_Per_Sec
- DbTime
- Exec_Per_Sec
- Hang
- LFParaWrite_Per_Sec
- LFSync_Per_Sec
- Logic_Read_Per_Sec
- Login_Per_Sec
- MEM_Total
- MEM_Used
- MEM_Used_Pct
- MEM_real_util
- On_Off_State
- Physical_Read_Per_Sec
- Proc_Used_Pct
- Proc_User_Used_Pct
- Redo_Per_Sec
- Row_Lock
- SctRead_Per_Sec
- SeqRead_Per_Sec
- Sess_Active
- Sess_Connect
- Session_pct
- TPS_Per_Sec
```

In [ ]:
# Step 12: match_observation(Component anomaly overview (ranked):
docker_002  metric_anoms=0 (top: n/a sev=0.0)  trace_anoms=21  log_anoms=0
os_021  
...[truncated])

```
ObAgent: uncertain.

Analysis: high Disk_await/Disk_io_util on multiple OS nodes and elevated Redo_Per_Sec on DB nodes point to I/O-heavy DB activity causing broad trace anomalies, not a clear network or CPU signature. To disambiguate, run:
- whether_is_abnormal_metric(Disk_await, Disk_io_util) on os_021/os_018,
- whether_is_abnormal_metric(Redo_Per_Sec) on db_007/db_003,
- whether_is_abnormal_metric(Login_Per_Sec and DB connection-count / failed-connections) and inspect DB error logs for connection-limit/close errors,
- whether_is_abnormal_metric(TCP_Retransmits or RTT) or collect_trace on service→DB spans to locate time spent (network vs storage vs DB).
JudgeAgent: No - The agents concluded storage I/O / DB performance issues (e.g., db_007 DbTime/Exec_Per_Sec spikes at 2020-05-27 01:12:00 and broad OS disk KPI anomalies and trace latency spikes ~2020-05-27 01:27:00), but this is not one of the allowed anomaly types (CPU fault, network delay, network loss, db connection limit, db close). Also the location was not pinned to exactly one component (multiple os_* and db_* candidates), and the occurrence minute is ambiguous (01:12 vs 01:27). Therefore the root cause was not correctly and completely identified.
```